# Phase 4: Feature Engineering and Model-Ready Dataset

## Objective

Create leakage-conscious features for forecasting industrial energy consumption 15 minutes and 1 hour ahead.

This notebook:
1. Loads and validates the Phase 2 dataset.
2. Creates timestamp-derived features and historical energy-use lags.
3. Creates historical lagged sensor features.
4. Defines three feature sets: X_A, X_B, and X_C.
5. Audits missing values and target leakage.
6. Builds a common modeling sample for fair feature-set comparison.
7. Saves the model-ready dataset and feature configuration.

## Forecasting Targets

- `target_15min`: energy consumption one 15-minute step ahead.
- `target_1hour`: energy consumption four 15-minute steps ahead.

## Leakage Policy

Target columns must never be used as predictors.

Current sensor values in X_C are included as an experimental feature set. Their availability at the forecast origin must be verified before interpreting X_C as operationally deployable.

In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

DATA_DIR = Path("../Data/processed")

INPUT_PATH = DATA_DIR / "steel_industry_phase2_targets.csv"
MODEL_READY_PATH = DATA_DIR / "steel_industry_model_ready.csv"
FEATURE_CONFIG_PATH = DATA_DIR / "feature_config.json"

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_PATH.resolve()}\n"
        "Confirm that Phase 2 saved steel_industry_phase2_targets.csv "
        "inside Data/processed and that the notebook is running from Notebooks/."
    )

print("Input:", INPUT_PATH.resolve())
print("Output directory:", DATA_DIR.resolve())

Input: D:\Research Paper Codes\Data\processed\steel_industry_phase2_targets.csv
Output directory: D:\Research Paper Codes\Data\processed


## 2. Load and Validate the Phase 2 Dataset

This cell checks:
- Required columns
- Chronological ordering
- Unique timestamps
- Regular 15-minute intervals
- Correct future target shifts

The validation must pass before feature engineering continues.

In [3]:
df = pd.read_csv(INPUT_PATH, parse_dates=["timestamp"])

df = (
    df.sort_values("timestamp")
      .reset_index(drop=True)
)

required_columns = [
    "timestamp",
    "Usage_kWh",
    "target_15min",
    "target_1hour",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

assert df["timestamp"].notna().all(), (
    "Some timestamps could not be parsed."
)

assert df["timestamp"].is_monotonic_increasing, (
    "Timestamps are not sorted."
)

assert df["timestamp"].is_unique, (
    "Duplicate timestamps found."
)

expected_interval = pd.Timedelta(minutes=15)

intervals = df["timestamp"].diff().dropna()

gap_count = int(
    (intervals != expected_interval).sum()
)

print("Dataset shape:", df.shape)

print(
    "Time range:",
    df["timestamp"].min(),
    "to",
    df["timestamp"].max()
)

print("Non-15-minute intervals:", gap_count)

print(
    "Missing target_15min:",
    df["target_15min"].isna().sum()
)

print(
    "Missing target_1hour:",
    df["target_1hour"].isna().sum()
)

if gap_count:
    raise ValueError(
        "Irregular timestamp intervals detected. "
        "Resolve or explicitly handle gaps before using "
        "row shifts as forecasting horizons."
    )

expected_target_15min = df["Usage_kWh"].shift(-1)
expected_target_1hour = df["Usage_kWh"].shift(-4)

if not df["target_15min"].equals(expected_target_15min):
    raise ValueError(
        "target_15min does not match Usage_kWh.shift(-1). "
        "Check Phase 2."
    )

if not df["target_1hour"].equals(expected_target_1hour):
    raise ValueError(
        "target_1hour does not match Usage_kWh.shift(-4). "
        "Check Phase 2."
    )

print("Timestamp and target validation passed.")

Dataset shape: (35040, 16)
Time range: 2018-01-01 00:00:00 to 2018-12-31 23:45:00
Non-15-minute intervals: 0
Missing target_15min: 1
Missing target_1hour: 4
Timestamp and target validation passed.


## 3. Create Calendar Features and Historical Energy-Use Lags

Calendar features are derived from the timestamp.

Historical usage lags contain only observations at or before the forecast origin.

- `usage_lag_1`: previous 15-minute reading
- `usage_lag_2`: previous 30-minute reading
- `usage_lag_4`: previous 1-hour reading
- `usage_lag_96`: same time on the previous day

In [4]:
df["hour"] = df["timestamp"].dt.hour

df["minute"] = df["timestamp"].dt.minute

df["day_of_week_num"] = (
    df["timestamp"].dt.dayofweek
)

df["month"] = df["timestamp"].dt.month

for lag in [1, 2, 4, 96]:
    df[f"usage_lag_{lag}"] = (
        df["Usage_kWh"].shift(lag)
    )

features_A = [
    "usage_lag_1",
    "usage_lag_2",
    "usage_lag_4",
    "usage_lag_96",
    "hour",
    "minute",
    "day_of_week_num",
    "month",
]

print("X_A feature count:", len(features_A))

display(
    df[
        ["timestamp", "Usage_kWh"] + features_A
    ].head(10)
)

X_A feature count: 8


,timestamp,Usage_kWh,usage_lag_1,usage_lag_2,usage_lag_4,usage_lag_96,hour,minute,day_of_week_num,month
0,2018-01-01 00:00:00,3.42,NaN,NaN,NaN,NaN,0,0,0,1
1,2018-01-01 00:15:00,3.17,3.42,NaN,NaN,NaN,0,15,0,1
2,2018-01-01 00:30:00,4.00,3.17,3.42,NaN,NaN,0,30,0,1
3,2018-01-01 00:45:00,3.24,4.00,3.17,NaN,NaN,0,45,0,1
4,2018-01-01 01:00:00,3.31,3.24,4.00,3.42,NaN,1,0,0,1
5,2018-01-01 01:15:00,3.82,3.31,3.24,3.17,NaN,1,15,0,1
6,2018-01-01 01:30:00,3.28,3.82,3.31,4.00,NaN,1,30,0,1
7,2018-01-01 01:45:00,3.60,3.28,3.82,3.24,NaN,1,45,0,1
8,2018-01-01 02:00:00,3.60,3.60,3.28,3.31,NaN,2,0,0,1
9,2018-01-01 02:15:00,3.28,3.60,3.60,3.82,NaN,2,15,0,1


## 4. Validate Feature Set X_A

The first 96 rows will not have a complete `usage_lag_96`.

These rows are expected to be unavailable for modeling with X_A.

In [5]:
lag_columns = [
    "usage_lag_1",
    "usage_lag_2",
    "usage_lag_4",
    "usage_lag_96",
]

print("Missing values in energy-use lags:")

display(
    df[lag_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

assert not any(
    target in features_A
    for target in ["target_15min", "target_1hour"]
)

assert "Usage_kWh" not in features_A

assert all(
    col in df.columns
    for col in features_A
)

print("X_A validation passed.")

Missing values in energy-use lags:


,missing_count
usage_lag_1,1
usage_lag_2,2
usage_lag_4,4
usage_lag_96,96


X_A validation passed.


## 5. Create Historical Sensor Features for X_B

X_B adds one-step lagged electrical measurements.

These features use the previous row rather than the current row, reducing the risk of using sensor readings that arrive after the forecast is issued.

In [7]:
sensor_columns = [
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
]

for column in sensor_columns:
    df[f"{column}_lag_1"] = (
        df[column].shift(1)
    )

sensor_lag_columns = [
    f"{column}_lag_1"
    for column in sensor_columns
]

features_B = features_A + sensor_lag_columns

print("X_A feature count:", len(features_A))

print("X_B feature count:", len(features_B))

print("X_B features:")

print(features_B)

display(
    df[
        ["timestamp", "Usage_kWh"] + features_B
    ].head(5)
)

X_A feature count: 8
X_B feature count: 12
X_B features:
['usage_lag_1', 'usage_lag_2', 'usage_lag_4', 'usage_lag_96', 'hour', 'minute', 'day_of_week_num', 'month', 'Lagging_Current_Reactive.Power_kVarh_lag_1', 'Leading_Current_Reactive_Power_kVarh_lag_1', 'Lagging_Current_Power_Factor_lag_1', 'Leading_Current_Power_Factor_lag_1']


,timestamp,Usage_kWh,usage_lag_1,usage_lag_2,usage_lag_4,usage_lag_96,hour,minute,day_of_week_num,month,Lagging_Current_Reactive.Power_kVarh_lag_1,Leading_Current_Reactive_Power_kVarh_lag_1,Lagging_Current_Power_Factor_lag_1,Leading_Current_Power_Factor_lag_1
0,2018-01-01 00:00:00,3.42,NaN,NaN,NaN,NaN,0,0,0,1,NaN,NaN,NaN,NaN
1,2018-01-01 00:15:00,3.17,3.42,NaN,NaN,NaN,0,15,0,1,3.46,0.0,70.30,100.0
2,2018-01-01 00:30:00,4.00,3.17,3.42,NaN,NaN,0,30,0,1,2.95,0.0,73.21,100.0
3,2018-01-01 00:45:00,3.24,4.00,3.17,NaN,NaN,0,45,0,1,4.46,0.0,66.77,100.0
4,2018-01-01 01:00:00,3.31,3.24,4.00,3.42,NaN,1,0,0,1,3.28,0.0,70.28,100.0


## 6. Validate Feature Set X_B

Check that all historical sensor features exist and that no forecasting target appears in the predictor list.

In [8]:
assert all(
    col in df.columns
    for col in features_B
)

assert not any(
    target in features_B
    for target in ["target_15min", "target_1hour"]
)

print("Missing values in lagged sensor features:")

display(
    df[sensor_lag_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

print("X_B validation passed.")

Missing values in lagged sensor features:


,missing_count
Lagging_Current_Reactive.Power_kVarh_lag_1,1
Leading_Current_Reactive_Power_kVarh_lag_1,1
Lagging_Current_Power_Factor_lag_1,1
Leading_Current_Power_Factor_lag_1,1


X_B validation passed.


## 7. Define Feature Set X_C Using Current Sensor Readings

X_C adds contemporaneous sensor values to X_B.

These measurements may improve predictive accuracy, but their inclusion is operationally valid only if they are available at the exact forecast origin.

Treat X_C as a conditional experimental feature set until this is confirmed.

Do not include `CO2(tCO2)` or `Load_Type` in the initial feature sets. Their derivation or availability at forecast time has not been established.

In [9]:
features_C = features_B + sensor_columns

print("X_A feature count:", len(features_A))

print("X_B feature count:", len(features_B))

print("X_C feature count:", len(features_C))

print("Additional X_C features:", sensor_columns)

assert all(
    col in df.columns
    for col in features_C
)

assert not any(
    target in features_C
    for target in ["target_15min", "target_1hour"]
)

print("X_C feature definition passed.")

X_A feature count: 8
X_B feature count: 12
X_C feature count: 16
Additional X_C features: ['Lagging_Current_Reactive.Power_kVarh', 'Leading_Current_Reactive_Power_kVarh', 'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor']
X_C feature definition passed.


## 8. Compare Feature-Set Completeness and Check for Leakage

Compare the number of features and missing values for each feature set.

The target columns must never appear in the predictor lists.

In [10]:
feature_sets = {
    "X_A": features_A,
    "X_B": features_B,
    "X_C": features_C,
}

audit_rows = []

for name, features in feature_sets.items():

    audit_rows.append({
        "feature_set": name,
        "feature_count": len(features),
        "missing_feature_cells": int(
            df[features].isna().sum().sum()
        ),
        "contains_target_column": any(
            col in features
            for col in [
                "target_15min",
                "target_1hour"
            ]
        ),
    })

feature_audit = pd.DataFrame(audit_rows)

display(feature_audit)

assert not feature_audit[
    "contains_target_column"
].any()

print(
    "No target columns are included "
    "in the feature definitions."
)

,feature_set,feature_count,missing_feature_cells,contains_target_column
0,X_A,8,103,False
1,X_B,12,107,False
2,X_C,16,107,False


No target columns are included in the feature definitions.


## 9. Create a Common Modeling Sample

For a fair comparison between X_A, X_B, and X_C, all three feature sets use the same rows.

The sample requires all X_C features and both targets to be present. This may discard more rows than fitting each feature set on its own eligible sample, but it ensures the comparison uses identical observations.

The longest lag (`usage_lag_96`) makes the earliest 96 rows ineligible. The final four rows should lack the one-hour target. Other missing values, if any, will also be excluded and reported.

In [11]:
required_model_columns = (
    features_C
    + ["target_15min", "target_1hour"]
)

missing_before = (
    df[required_model_columns]
    .isna()
    .sum()
)

df_model = (
    df.dropna(subset=required_model_columns)
      .copy()
      .reset_index(drop=True)
)

print(
    "Rows before eligibility filtering:",
    len(df)
)

print(
    "Rows in common modeling sample:",
    len(df_model)
)

print(
    "Rows excluded:",
    len(df) - len(df_model)
)

print(
    "\nMissing values before filtering "
    "(non-zero only):"
)

display(
    missing_before[missing_before > 0]
    .to_frame("missing_count")
)

for name, features in feature_sets.items():

    assert df_model[features].notna().all().all(), (
        f"Missing features remain in {name}"
    )

assert df_model[
    ["target_15min", "target_1hour"]
].notna().all().all()

assert df_model["timestamp"].is_monotonic_increasing

assert df_model["timestamp"].is_unique

print("Common modeling sample validation passed.")

print("Final modeling shape:", df_model.shape)

print(
    "Modeling time range:",
    df_model["timestamp"].min(),
    "to",
    df_model["timestamp"].max()
)

Rows before eligibility filtering: 35040
Rows in common modeling sample: 34940
Rows excluded: 100

Missing values before filtering (non-zero only):


,missing_count
usage_lag_1,1
usage_lag_2,2
usage_lag_4,4
usage_lag_96,96
Lagging_Current_Reactive.Power_kVarh_lag_1,1
Leading_Current_Reactive_Power_kVarh_lag_1,1
Lagging_Current_Power_Factor_lag_1,1
Leading_Current_Power_Factor_lag_1,1
target_15min,1
target_1hour,4


Common modeling sample validation passed.
Final modeling shape: (34940, 28)
Modeling time range: 2018-01-02 00:00:00 to 2018-12-31 22:45:00


## 10. Save the Model-Ready Dataset and Feature Configuration

Save the model-ready dataset and feature configuration in `Data/processed/`.

The feature configuration records which columns belong to each feature set and documents the remaining availability assumptions.

In [12]:
DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df_model.to_csv(
    MODEL_READY_PATH,
    index=False
)

feature_config = {
    "X_A": features_A,
    "X_B": features_B,
    "X_C": features_C,

    "targets": [
        "target_15min",
        "target_1hour"
    ],

    "forecast_horizons": {
        "target_15min": "15 minutes ahead",
        "target_1hour": "60 minutes ahead"
    },

    "notes": {
        "X_A": (
            "Historical energy-use lags "
            "and timestamp-derived calendar features."
        ),

        "X_B": (
            "X_A plus one-step lagged "
            "electrical sensor measurements."
        ),

        "X_C": (
            "X_B plus contemporaneous sensor measurements; "
            "use only if available at forecast origin."
        ),

        "excluded_initially": [
            (
                "CO2(tCO2): derivation and timing unverified"
            ),
            (
                "Load_Type: assignment rule and "
                "forecast-time availability unverified"
            )
        ]
    }
}

with open(
    FEATURE_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        feature_config,
        file,
        indent=4
    )

print(
    "Saved model-ready dataset:",
    MODEL_READY_PATH.resolve()
)

print(
    "Saved feature configuration:",
    FEATURE_CONFIG_PATH.resolve()
)

print("Final shape:", df_model.shape)

Saved model-ready dataset: D:\Research Paper Codes\Data\processed\steel_industry_model_ready.csv
Saved feature configuration: D:\Research Paper Codes\Data\processed\feature_config.json
Final shape: (34940, 28)
